# Іnstalling Libraries

In [1]:
!pip install transformers datasets evaluate numpy rouge_score accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.5 MB/s eta 0:00:00


# Setup & Data Loading

In [2]:
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

train_dataset = load_dataset("cnn_dailymail", '3.0.0', split="train").shuffle(seed=42).select(range(10000))
eval_dataset = load_dataset("cnn_dailymail", '3.0.0', split="validation").shuffle(seed=42).select(range(1000))

MODEL_NAME = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
print(f"Model {MODEL_NAME} loaded")

README.md: 0.00B [00:00, ?B/s]

3.0.0/train-00000-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00001-of-00003.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

3.0.0/train-00002-of-00003.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

3.0.0/validation-00000-of-00001.parquet:   0%|          | 0.00/34.7M [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Model facebook/bart-base loaded


# Data Cleaning / Text Preprocessing

In [3]:
import re

# 1. Залишаємо базову функцію без змін
import re

def clean_text(text):
    if not isinstance(text, str):
        return str(text)
        
    text = re.sub(r"^.*?UPDATED:.*?\d{4}\s*\.\s*", "", text)
    text = re.sub(r"^.*?PUBLISHED:.*?\d{4}\s*\.\s*", "", text)
    
    text = re.sub(r"\s+\.\s+", ". ", text)

    text = re.sub(r'[\n\r\t]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def clean_hf_dataset(examples):
    examples['article'] = [clean_text(text) for text in examples['article']]
    examples['highlights'] = [clean_text(text) for text in examples['highlights']]
    return examples

train_dataset = train_dataset.map(clean_hf_dataset, batched=True)
eval_dataset = eval_dataset.map(clean_hf_dataset, batched=True)

train_dataset = train_dataset.remove_columns(["id"])
eval_dataset = eval_dataset.remove_columns(["id"])

print("Data is clear (article та highlights)!")

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Data is clear (article та highlights)!


# Tokenization & Metrics Setup

In [4]:
def preprocess_function(examples):
    inputs = [doc for doc in examples["article"]]
    model_inputs = tokenizer(inputs, max_length=1024, truncation=True)

    labels = tokenizer(text_target=examples["highlights"], max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train_dataset = train_dataset.map(preprocess_function, batched=True)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True)

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return {k: round(v, 4) for k, v in result.items()}

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

# Fine-tuning Pipeline & Saving

In [5]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

for key in ["max_length", "num_beams", "no_repeat_ngram_size"]:
    if hasattr(model.config, key):
        delattr(model.config, key)

model.generation_config.no_repeat_ngram_size = 3
model.generation_config.max_length = 128
model.generation_config.num_beams = 4

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/summarizer_results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    weight_decay=0.01,
    num_train_epochs=3,
    predict_with_generate=True,
    logging_strategy="steps",
    logging_steps=100,
    generation_num_beams=4,
    generation_max_length=128,
    fp16=True 
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_eval_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train(resume_from_checkpoint=False)

model_save_path = "/kaggle/working/my_finetuned_summarizer"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)
print("Training is over.")

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,4.397163,1.827199,0.422500,0.201500,0.290200,0.289800
2,4.111225,1.793178,0.426300,0.202500,0.292500,0.292400
3,4.016707,1.793373,0.426600,0.203000,0.292700,0.292500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training is over.


# Show Data

In [6]:
import pandas as pd
print(train_dataset)
print(train_dataset.column_names)

df_sample = pd.DataFrame(train_dataset[:502])
display(df_sample)

Dataset({
    features: ['article', 'highlights'],
    num_rows: 10000
})
['article', 'highlights']


,article,highlights
0,Three members of the same family who died in a...,John and. Audrey Cook were discovered alongsid...
1,UNITED NATIONS (CNN) -- A rare meeting of U.N....,NEW: Libya can serve as example of cooperation...
2,Cover-up: Former Archbishop Lord Hope allowed ...,Very Reverend Robert Waddington sexually abuse...
3,TLC has pulled an episode of Cake Boss from fu...,Monday night's episode showed Buddy Valastro t...
4,'The lamps are going out all over Europe. We s...,People asked to turn out lights for hour betwe...
...,...,...
497,(CNN) -- Being coach of a nation's football te...,Roy Hodgson takes charge of England team for f...
498,Literary giant Sir Salman Rushdie is romancing...,"The Midnight's Children's scribe, 65, has four..."
499,Five people have been killed after a gunman we...,"Shayne Riggleman, 22, murdered five people, in..."
500,Myanmar (CNN) -- When Nyein Chan Aung sold his...,People here completely bypass the banks and ke...


# Sanity Check & Final Evaluation

In [7]:
# final_metrics = trainer.evaluate()

# print("Final metrics")
# for key, value in final_metrics.items():
#     print(f"{key}: {value}")

# Test Model

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import os

model_path = "/kaggle/working/my_finetuned_summarizer"

if not os.path.exists(model_path):
    print(f"Error: Folder {model_path} not found!")
else:
    print("Folder is found!\n")

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)

def summarize_text(text):
    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt")
    
    summary_ids = model.generate(
        inputs["input_ids"],
        max_length=128,
        min_length=20,
        length_penalty=2.0,
        num_beams=4,
        no_repeat_ngram_size=3,
        early_stopping=True
    )
    
    output = tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    
    clean_output = output.replace("Â", "").replace("\xa0", " ").replace("", "")
    
    import re
    clean_output = re.sub(r'\s+', ' ', clean_output).strip()
    
    return clean_output

long_article = """
The James Webb Space Telescope (JWST) has discovered a massive, ancient galaxy that shouldn't exist according to current cosmological models. 
Astronomers analyzing the latest infrared data found the galaxy, designated ZF-UDS-7329, which formed just 800 million years after the Big Bang. 
What puzzles scientists is that the galaxy contains more mass in stars than the Milky Way, despite its incredibly young age. 
Standard theories of galaxy formation suggest that it takes billions of years for dark matter halos to pull together enough gas and dust to form such massive stellar populations. 
"This completely upends our understanding of how quickly galaxies can form," said Dr. Sarah Jenkins, lead researcher on the project. 
The team is now requesting more observation time to analyze the galaxy's chemical composition. 
If confirmed, this discovery might force astrophysicists to rewrite the timelines of the early universe and rethink the fundamental nature of dark matter's influence on cosmic evolution.
"""


print("Full article")
print(long_article.strip())
print("\n-Summarize article")
print(summarize_text(long_article))

# Push in HF

In [10]:
from huggingface_hub import login

login("TOKEN_HERE")

hf_repo_name = "YKostiantyn/fine-tuned-bart-model-summarizer"

print(f"Start downloading to the repository: {hf_repo_name}...")

model.push_to_hub(hf_repo_name)
tokenizer.push_to_hub(hf_repo_name)

print("The model was downloaded successfully.")

Start downloading to the repository: YKostiantyn/fine-tuned-bart-model-summarizer...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

The model was downloaded successfully.
